# 12 · Lista de casa: SciPy

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

## Instruções

- Trabalhe em uma cópia deste notebook e substitua cada `...` pelo seu código.
- O alvo desta lista é "a avaliação recebeu **nota 5**".
- Cerca de metade da nota está nas respostas escritas. Use os números que você obteve.
- Antes de entregar, use **Reiniciar sessão e executar tudo**.

| Questão | Tema | Pontos |
|---|---|---|
| 1 | Intervalos para amostras pequenas | 2,0 |
| 2 | Sete testes e uma conclusão | 2,5 |
| 3 | Ajuste de curva e extrapolação | 2,5 |
| 4 | Busca por semelhança, sem sair do formato esparso | 2,0 |
| 5 | Que ferramenta usar | 1,0 |

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, sparse, stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

SEMENTE = 42
URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

df = pd.read_csv(ARQUIVO, dtype={"product_id": str}, low_memory=False)
df["nota_5"] = (df["overall_rating"] == 5).astype(int)
df["texto"] = df["review_title"].fillna("") + ". " + df["review_text"].fillna("")
df["dia_semana"] = pd.to_datetime(df["submission_date"]).dt.dayofweek          # 0 = segunda-feira ... 6 = domingo
print(f"{len(df)} avaliações · nota 5 em {df['nota_5'].mean():.1%}")

## Questão 1 (2,0 pontos): intervalos para amostras pequenas

A tabela `estados` tem, para cada estado, o número de avaliações (`n`) e o número de notas 5 (`k`).

a) Acrescente as colunas `taxa`, `wald_inf`, `wald_sup` (intervalo de Wald de 95%, pela fórmula) e `wilson_inf`, `wilson_sup` (intervalo de Wilson de 95%, com `stats.binomtest`).

b) Guarde em `menores_q1` as linhas dos **cinco estados com menos avaliações**, ordenadas por `n` crescente.

In [ ]:
estados = df.groupby("reviewer_state")["nota_5"].agg(n="size", k="sum")

In [ ]:
# ✏️ Questão 1
# SEU CÓDIGO AQUI
estados["taxa"] = ...
...
menores_q1 = ...

In [ ]:
# ✅ Conferência da questão 1 (não altere)
assert {"taxa", "wald_inf", "wald_sup", "wilson_inf", "wilson_sup"} <= set(estados.columns)
assert len(menores_q1) == 5 and menores_q1["n"].is_monotonic_increasing and menores_q1["n"].iloc[0] == estados["n"].min()
assert ((estados["wald_inf"] < estados["taxa"]) & (estados["taxa"] < estados["wald_sup"])).all()
assert ((estados["wilson_inf"] < estados["taxa"]) & (estados["taxa"] < estados["wilson_sup"])).all()
_sp = estados.loc["SP"]
assert abs((_sp["wald_sup"] - _sp["wald_inf"]) - (_sp["wilson_sup"] - _sp["wilson_inf"])) < 1e-3, "em um estado grande, as duas larguras devem praticamente coincidir"
print("Formato correto.")

**1c)** Nos cinco estados menores, qual é a largura dos intervalos? Algum deles permite afirmar que a taxa do estado é diferente da taxa do país? Em que os intervalos de Wald e de Wilson diferem aqui, e qual você usaria em um relatório?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 2 (2,5 pontos): sete testes e uma conclusão

Pergunta: as notas dadas dependem do **dia da semana** em que a avaliação foi enviada?

a) Para cada dia da semana (0 a 6), compare as notas (`overall_rating`) daquele dia com as de **todos os outros dias**, com o teste de Mann-Whitney. Monte `tabela_q2`, indexada pelo dia, com as colunas `n` (avaliações no dia), `nota_media`, `diferenca` (nota média do dia menos a dos outros dias) e `p_valor`.

b) Acrescente a coluna `significante`: `True` quando o p-valor é menor que o limite de 0,05 **corrigido por Bonferroni** para sete testes.

In [ ]:
# ✏️ Questão 2
# SEU CÓDIGO AQUI
tabela_q2 = ...

In [ ]:
# ✅ Conferência da questão 2 (não altere)
assert list(tabela_q2.index) == list(range(7)) and list(tabela_q2.columns) == ["n", "nota_media", "diferenca", "p_valor", "significante"]
assert tabela_q2["n"].sum() == len(df) and tabela_q2["p_valor"].between(0, 1).all()
_d0 = df.loc[df["dia_semana"] == 0, "overall_rating"].mean() - df.loc[df["dia_semana"] != 0, "overall_rating"].mean()
assert abs(tabela_q2.loc[0, "diferenca"] - _d0) < 1e-9, "a diferença é a média do dia menos a média dos outros dias"
assert (tabela_q2["significante"] == (tabela_q2["p_valor"] < 0.05 / 7)).all()
print(f"Formato correto: {int(tabela_q2['significante'].sum())} de 7 dias 'significantes' depois da correção; maior diferença absoluta: {tabela_q2['diferenca'].abs().max():.3f} ponto.")

**2c)** Quantos dias são "significantes" depois da correção? Qual é a maior diferença, em pontos de nota e em desvios padrão? Que conclusão você levaria a um gestor: as notas dependem do dia da semana?

**2d)** Por que a correção para múltiplos testes é necessária aqui? Se, em vez de sete testes planejados, você tivesse procurado diferenças em dezenas de recortes (dia, hora, estado, categoria) e relatado só as significantes, o que estaria errado?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 3 (2,5 pontos): ajuste de curva e extrapolação

A célula abaixo mede a AUC de um TF-IDF com regressão logística, para o alvo "nota 5", com quantidades crescentes de avaliações de treino.

a) Ajuste, com `optimize.curve_fit`, a curva `teto − b · n^(−c)` aos pontos medidos (`tamanhos`, `aucs`). Guarde os três parâmetros em `parametros_q3` e as incertezas (raiz da diagonal da matriz de covariância) em `incertezas_q3`.

b) Calcule `previsto_q3`: a AUC que a curva prevê para o conjunto de treino completo (`n_total`). Depois execute a célula que mede o valor real.

In [ ]:
treino, valid = train_test_split(df, test_size=0.2, stratify=df["nota_5"], random_state=SEMENTE)
embaralhado = treino.sample(frac=1, random_state=0)
n_total = len(embaralhado)


def auc_com(n):
    m = make_pipeline(TfidfVectorizer(min_df=1 if n < 1000 else 2), LogisticRegression(max_iter=1000))
    m.fit(embaralhado["texto"].iloc[:n], embaralhado["nota_5"].iloc[:n])
    return roc_auc_score(valid["nota_5"], m.predict_proba(valid["texto"])[:, 1])


tamanhos = [100, 300, 1000, 3000, 10000, 30000]
aucs = [auc_com(n) for n in tamanhos]
print(dict(zip(tamanhos, np.round(aucs, 4))))

In [ ]:
# ✏️ Questão 3
def curva(n, teto, b, c):
    return teto - b * n ** (-c)


# SEU CÓDIGO AQUI
parametros_q3 = ...
incertezas_q3 = ...
previsto_q3 = ...

In [ ]:
# ✅ Conferência da questão 3 (não altere)
assert len(parametros_q3) == 3 and len(incertezas_q3) == 3 and (np.asarray(incertezas_q3) > 0).all()
assert np.abs(curva(np.array(tamanhos), *parametros_q3) - np.array(aucs)).max() < 0.01, "a curva não passa perto dos pontos medidos"
assert abs(previsto_q3 - curva(n_total, *parametros_q3)) < 1e-12
real_q3 = auc_com(n_total)
print(f"Formato correto. Com {n_total} avaliações: previsto {previsto_q3:.4f} · medido {real_q3:.4f} · teto estimado {parametros_q3[0]:.4f} ± {incertezas_q3[0]:.4f}")

**3c)** A previsão para o conjunto completo ficou acima ou abaixo do valor medido? O valor medido está dentro da faixa `teto ± 2 incertezas`? O que isso ensina sobre extrapolar uma curva ajustada?

**3d)** Um gestor pergunta: "vale a pena pagar para rotular mais 100 mil avaliações?". Responda com base na curva e no valor medido, dizendo quanto de AUC você espera ganhar e que outra ação poderia render mais.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 4 (2,0 pontos): busca por semelhança, sem sair do formato esparso

A matriz `X` abaixo é o TF-IDF de 30 mil avaliações; suas linhas têm norma 1.

a) Escreva `mais_parecidas(i, k=5)`: devolve um DataFrame com as colunas `posicao`, `semelhanca` e `texto` das `k` avaliações mais parecidas com a avaliação `i` (semelhança de cosseno), **sem incluir a própria** e **sem converter `X` em matriz densa** (o vetor de semelhanças, com um número por avaliação, pode ser denso).

b) Calcule `mb_esparsa_q4` e `gb_densa_q4`: a memória ocupada por `X` (soma dos três vetores do formato CSR, em MB) e a que uma matriz densa do mesmo tamanho ocuparia, com 8 bytes por elemento (em GB).

In [ ]:
amostra_q4 = df.sample(30_000, random_state=SEMENTE).reset_index(drop=True)
X = TfidfVectorizer(min_df=3).fit_transform(amostra_q4["texto"])
print(type(X).__name__, X.shape)

In [ ]:
# ✏️ Questão 4
def mais_parecidas(i, k=5):
    # SEU CÓDIGO AQUI
    ...


# SEU CÓDIGO AQUI
mb_esparsa_q4 = ...
gb_densa_q4 = ...

In [ ]:
# ✅ Conferência da questão 4 (não altere)
_r = mais_parecidas(7, k=5)
assert list(_r.columns) == ["posicao", "semelhanca", "texto"] and len(_r) == 5 and 7 not in _r["posicao"].values, "cinco linhas, sem a própria avaliação"
assert _r["semelhanca"].is_monotonic_decreasing and _r["semelhanca"].between(0, 1 + 1e-9).all()
_j = int(_r["posicao"].iloc[0])
assert abs(_r["semelhanca"].iloc[0] - X[7].multiply(X[_j]).sum()) < 1e-9, "a semelhança deve ser o produto interno das duas linhas"
assert abs(mb_esparsa_q4 - (X.data.nbytes + X.indices.nbytes + X.indptr.nbytes) / 1e6) < 1e-6 and abs(gb_densa_q4 - X.shape[0] * X.shape[1] * 8 / 1e9) < 1e-9
print(f"Formato correto: {mb_esparsa_q4:.1f} MB no formato esparso contra {gb_densa_q4:.1f} GB em formato denso.")

**4c)** Escolha uma avaliação, mostre as cinco mais parecidas e comente: elas tratam do mesmo assunto? Que tipo de avaliação parecida esta busca **não** encontraria, e por quê? (Lembre da busca semântica da aula 09.)

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 5 (1,0 ponto): que ferramenta usar

a) Para cada situação, indique a função ou o módulo do SciPy que você usaria, em uma frase:

1. Estimar o intervalo de confiança da **AUC** de um modelo em um conjunto de teste de 800 exemplos.
2. Encontrar a temperatura `T` que, dividindo os *logits* de uma rede, minimiza a perda logarítmica na validação.
3. Verificar se o tempo de entrega (em dias) difere entre duas transportadoras, sabendo que a distribuição é muito assimétrica.
4. Guardar uma matriz de 2 milhões de clientes por 50 mil produtos, com 1 onde houve compra.

b) Declare como usou assistentes de IA nesta lista: em quais questões, para quê, e o que precisou corrigir.

✏️ **Sua resposta:**

_Escreva aqui._